# 6주차 실습. Single-cell RNA-seq

이 파일을 직접 고치지 말고 먼저 본인 폴더로 복사한다. Qoka 대화 입력창에 입력한다.

> course/labs/week6/week6_scrna.ipynb 를 analysis/week6/ 폴더로 복사해서 열어줘.

열리면 오른쪽 위 커널을 **Qoka Run Environment**로 고른다.

## 이 노트북을 쓰는 방법

- scanpy 튜토리얼 "Preprocessing and clustering 3k PBMCs"의 단계와 pseudobulk 차등발현을 셀 하나씩 나눔. 코드는 미리 적혀 있음
- 셀마다: **프롬프트**를 읽는다 → 코드에서 프롬프트의 말이 된 줄을 찾는다 → **예상**을 적는다 → 실행하고 비교한다. 결과 설명은 강의노트
- 셀은 위에서부터 순서대로 실행

| 실습 | 셀 | 내용 | 강의노트 |
|---|---|---|---|
| 1 | 1–6 | PBMC 3k: QC, 정규화, HVG, PCA | 6.4–6.7 |
| 2 | 7–11 | PBMC 3k: clustering, cell type | 6.8–6.9 |
| 3 | 12–16 | Kang 2018: 세포 단위 검정 vs pseudobulk | 6.10 |

## 준비

패키지를 설치하고 파일 위치를 찾는다. 그대로 실행한다. "restart the kernel"이 나오면 커널 재시작 후 처음부터.

In [ ]:
%pip install -q scanpy==1.11.4 leidenalg igraph pydeseq2==0.5.2 matplotlib

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io
from pydeseq2.dds import DeseqDataSet   # 5주차와 같은 DESeq2
from pydeseq2.ds import DeseqStats

warnings.filterwarnings("ignore", category=FutureWarning)   # 라이브러리 내부 경고 (5주차 5.5절)

# data/ 폴더가 나올 때까지 위로 올라가며 프로젝트 폴더를 찾음 (3주차와 같음)
root = Path.cwd()
while not (root / "data").exists() and root != root.parent:
    root = root.parent
data_dir = root / "data" / "week6"
out_dir = root / "results" / "week6"      # 결과를 저장할 곳
out_dir.mkdir(parents=True, exist_ok=True)

sc.settings.datasetdir = data_dir         # PBMC 3k를 받을 곳
sc.settings.verbosity = 0
print(root)

---
# 실습 1. PBMC 3k: QC와 정규화

## 셀 1. 데이터 읽기

**프롬프트**: > scanpy 튜토리얼의 PBMC 3k 데이터를 읽어줘.

**예상**: 행과 열은 각각 무엇이고 몇 개일까? (5주차 카운트 표: 샘플 4 × 유전자 17,642)

> (실행하기 전에 여기에 적는다)

In [ ]:
adata = sc.datasets.pbmc3k()      # 10x Genomics PBMC 3k. 처음 한 번 인터넷에서 받음 (5.6 MB)
adata.var_names_make_unique()     # 같은 이름의 유전자에 -1, -2를 붙임
adata                             # AnnData: 행(obs) = 세포, 열(var) = 유전자

## 셀 2. 0과 세포별 총 count

**프롬프트**: > 0인 값의 비율과 세포별 총 count 범위를 보여줘.

**예상**: 0인 칸은 몇 %일까? 총 count가 가장 큰 세포는 가장 작은 세포의 몇 배일까? (5주차 샘플 사이: 1.7배)

> (실행하기 전에 여기에 적는다)

In [ ]:
X = adata.X                                   # 희소 행렬: 0이 아닌 값만 저장
total = np.asarray(X.sum(axis=1)).ravel()     # 세포별 총 count
print(f"0의 비율: {1 - X.nnz / (X.shape[0] * X.shape[1]):.1%}")
print(f"세포별 총 count: 최소 {total.min():.0f}, 중앙값 {np.median(total):.0f}, 최대 {total.max():.0f} ({total.max() / total.min():.0f}배)")

## 셀 3. 미토콘드리아 비율

**프롬프트**: > 미토콘드리아 유전자 비율을 계산해줘.

**예상**: 접두사를 생쥐 표기 `mt-`로 쓰면 결과는? 오류가 날까? (강의노트 6.5절)

> (실행하기 전에 여기에 적는다)

In [ ]:
for prefix in ["MT-", "mt-"]:                 # 사람 표기, 생쥐 표기
    print(f"{prefix} 로 시작하는 유전자: {adata.var_names.str.startswith(prefix).sum()}개")

adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
print(f"미토콘드리아 비율 중앙값 {adata.obs.pct_counts_mt.median():.2f}%, 5% 이상인 세포 {(adata.obs.pct_counts_mt >= 5).sum()}개")
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)

## 셀 4. 세포와 유전자 거르기

**프롬프트**: > 튜토리얼 기준으로 세포와 유전자를 걸러줘.

**예상**: 세포는 몇 개 남을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)     # 검출 유전자 200개 미만 세포 제거 (빈 droplet)
sc.pp.filter_genes(adata, min_cells=3)       # 3개 미만의 세포에서만 검출된 유전자 제거

# 유전자가 너무 많은 세포(doublet 의심), 미토콘드리아 비율이 높은 세포(손상) 제거
adata = adata[(adata.obs.n_genes_by_counts < 2500) & (adata.obs.pct_counts_mt < 5)].copy()
adata.shape

## 셀 5. 세포별 정규화

**프롬프트**: > 세포마다 총 count를 맞추고 log 변환해줘.

**예상**: 정규화 후 세포별 총합은? 5주차 size factor와 다른 점은? (강의노트 6.6절)

> (실행하기 전에 여기에 적는다)

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)     # 세포마다 총합을 10,000으로
sc.pp.log1p(adata)                               # log(x + 1). 0은 0으로 남음
adata.raw = adata                                # 모든 유전자의 정규화 값 보관 (셀 11 marker 확인용)

after = np.asarray(np.expm1(adata.X).sum(axis=1)).ravel()   # log를 되돌려 총합 확인
print("정규화 후 세포 3개의 총합:", after[:3].round(0))

## 셀 6. HVG와 PCA

**프롬프트**: > HVG를 고르고 PCA를 해줘.

**예상**: PC1의 설명 비율은 5주차(57%)보다 클까 작을까? (강의노트 6.7절)

> (실행하기 전에 여기에 적는다)

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)   # 튜토리얼 기준
print("HVG:", adata.var.highly_variable.sum(), "개")

adata = adata[:, adata.var.highly_variable].copy()         # HVG만 남김
sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"])  # 총 count, 미토콘드리아 비율의 영향 제거 (튜토리얼 단계)
sc.pp.scale(adata, max_value=10)                            # 유전자마다 평균 0, 표준편차 1
sc.tl.pca(adata, svd_solver="arpack")
print("PC1–5 설명 비율 (%):", (adata.uns["pca"]["variance_ratio"][:5] * 100).round(1))
sc.pl.pca_variance_ratio(adata, log=True)

---
# 실습 2. PBMC 3k: clustering과 cell type

## 셀 7. 이웃 그래프와 UMAP

**프롬프트**: > 이웃 그래프와 UMAP을 만들어줘.

**예상**: 세포는 몇 덩어리로 보일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)   # 튜토리얼 값: 세포마다 가까운 세포 10개, PC 40개 사용
sc.tl.umap(adata)                                  # 이웃 관계를 2차원 그림으로
sc.pl.umap(adata, color=["CST3", "NKG7", "MS4A1"])

## 셀 8. Leiden clustering

**프롬프트**: > Leiden clustering을 resolution 0.5와 1.0으로 해줘.

**예상**: 각각 cluster는 몇 개일까? (교재 0.5, 튜토리얼 1.0. 강의노트 6.8절)

> (실행하기 전에 여기에 적는다)

In [ ]:
for res in [0.5, 1.0]:
    sc.tl.leiden(adata, resolution=res, random_state=0, key_added=f"leiden_{res}",
                 flavor="igraph", n_iterations=2, directed=False)
    print(f"resolution {res}: cluster {adata.obs[f'leiden_{res}'].nunique()}개")
sc.pl.umap(adata, color=["leiden_0.5", "leiden_1.0"], legend_loc="on data")

## 셀 9. seed 바꾸기

**프롬프트**: > resolution 1.0에서 seed만 바꿔 cluster 수를 비교해줘.

**예상**: seed(난수 시작값)만 바꾸면 cluster 수가 바뀔까?

> (실행하기 전에 여기에 적는다)

In [ ]:
for seed in [0, 1, 2]:
    sc.tl.leiden(adata, resolution=1.0, random_state=seed, key_added="tmp",
                 flavor="igraph", n_iterations=2, directed=False)
    sizes = adata.obs["tmp"].value_counts().tolist()
    print(f"seed {seed}: cluster {len(sizes)}개, 크기 {sizes}")
del adata.obs["tmp"]

## 셀 10. Marker 유전자

**프롬프트**: > cluster마다 marker 유전자를 보여줘.

**예상**: cluster 0의 1위 유전자는 cell type을 알려 줄까? (강의노트 6.9절)

> (실행하기 전에 여기에 적는다)

In [ ]:
# cluster 하나 vs 나머지 전체, 세포 단위 Wilcoxon 검정 (resolution 1.0, seed 0)
sc.tl.rank_genes_groups(adata, "leiden_1.0", method="wilcoxon")
pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(5)   # cluster마다 상위 5개

## 셀 11. Cell type 이름 붙이기

**프롬프트**: > 튜토리얼의 marker로 cluster에 이름을 붙여줘.

**예상**: marker 하나가 두 cluster에서 모두 높으면? (예: NKG7)

> (실행하기 전에 여기에 적는다)

In [ ]:
markers = {"CD4 T": ["IL7R"], "CD14 Mono": ["CD14", "LYZ"], "B": ["MS4A1"], "CD8 T": ["CD8A"],
           "NK": ["GNLY", "NKG7"], "FCGR3A Mono": ["FCGR3A", "MS4A7"], "DC": ["FCER1A", "CST3"],
           "Megakaryocyte": ["PPBP"]}                      # 튜토리얼의 marker 표
sc.pl.dotplot(adata, markers, groupby="leiden_1.0")      # 점 크기: 발현 세포 비율, 색: 평균 발현

# dotplot을 보고 정한 이름 (resolution 1.0, seed 0 기준)
names = {"0": "CD4 T", "1": "CD8 T", "2": "B", "3": "FCGR3A Mono",
         "4": "CD14 Mono", "5": "NK", "6": "DC", "7": "Megakaryocyte"}
adata.obs["cell_type"] = adata.obs["leiden_1.0"].map(names)
print(adata.obs["cell_type"].value_counts())
sc.pl.umap(adata, color="cell_type", legend_loc="on data")
adata.write(out_dir / "pbmc3k_annotated.h5ad")           # 실습 3.5에서 씀

---
# 실습 3. Kang 2018: 세포 단위 검정 vs pseudobulk

PBMC, IFN-β 자극 vs 대조. 기증자 8명의 세포를 섞어서 한 번에 측정 (GEO GSE96583, batch 2).

## 데이터 받기 (Qoka 대화 입력창)

> GEO GSE96583에 어떤 샘플과 파일이 있는지 알려줘.

> batch 2에 해당하는 파일(샘플 2개의 파일과 이름에 batch2가 붙은 파일)만 data/week6/kang2018/ 에 받아줘.

**예상**: 파일은 몇 개일까? 무엇으로 제대로 받았는지 확인할까? (강의노트 6.10절)

> (실행하기 전에 여기에 적는다)

## 셀 12. 10x 파일 읽기

**프롬프트**: > 받은 파일을 읽어서 singlet만 남겨줘.

**예상**: 세포 정보 표와 바코드 파일의 순서가 같을까? 기증자-조건마다 세포는 몇 개일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
kdir = data_dir / "kang2018"
for f in sorted(kdir.glob("*.gz")):
    print(f"{f.name:40s} {f.stat().st_size / 1e6:6.1f} MB")      # GEO 목록의 크기와 비교

genes = pd.read_csv(kdir / "GSE96583_batch2.genes.tsv.gz", sep="\t", header=None)          # Ensembl ID, symbol
info = pd.read_csv(kdir / "GSE96583_batch2.total.tsne.df.tsv.gz", sep="\t", index_col=0)   # 세포 정보

parts = []
for gsm, mtx, stim in [("GSM2560248", "2.1", "ctrl"), ("GSM2560249", "2.2", "stim")]:
    X = scipy.io.mmread(kdir / f"{gsm}_{mtx}.mtx.gz").T.tocsr()            # 파일은 유전자 × 세포 → 뒤집음
    barcodes = pd.read_csv(kdir / f"{gsm}_barcodes.tsv.gz", header=None)[0].values
    meta = info[info["stim"] == stim]                                       # 이 파일의 세포 정보 (같은 순서)
    same = (meta.index.str.replace(r"\.\d+$", "", regex=True) == barcodes).mean()
    print(f"{stim}: 세포 {len(barcodes)}개, 세포 정보와 바코드 일치 {same:.1%}")
    a = sc.AnnData(X.astype(np.float32))
    a.var_names = genes[1].values
    a.var_names_make_unique()
    a.obs_names = [f"{b}_{stim}" for b in barcodes]
    for col in ["ind", "stim", "cell", "multiplets"]:   # ind: 기증자, cell: 원 논문의 cell type
        a.obs[col] = meta[col].astype(str).values
    parts.append(a)
kang = sc.concat(parts, merge="same")

print(kang.obs["multiplets"].value_counts().to_dict())   # demuxlet 판정 (유전 변이로 기증자 구별)
kang = kang[(kang.obs["multiplets"] == "singlet") & (kang.obs["cell"] != "nan")].copy()
print("singlet:", kang.shape)
pd.crosstab(kang.obs["ind"], kang.obs["stim"])

## 셀 13. UMAP

**프롬프트**: > UMAP을 자극 여부, cell type, 기증자로 색칠해줘.

**예상**: 세포는 자극 여부로 갈릴까, cell type으로 갈릴까, 기증자로 갈릴까?

> (실행하기 전에 여기에 적는다)

In [ ]:
k = kang.copy()                                   # 그림용 사본. kang은 count 그대로 둠
sc.pp.filter_genes(k, min_cells=10)
sc.pp.normalize_total(k, target_sum=1e4)
sc.pp.log1p(k)
sc.pp.highly_variable_genes(k, n_top_genes=2000)
sc.pp.pca(k, n_comps=30, mask_var="highly_variable")
sc.pp.neighbors(k)
sc.tl.umap(k)
sc.pl.umap(k, color=["stim", "cell", "ind"], ncols=3, wspace=0.5)

## 셀 14. 세포 단위 검정

**프롬프트**: > CD14+ monocyte에서 자극과 대조를 세포 단위로 비교해줘.

**예상**: IFN 반응 유전자(ISG15 등)의 padj는? 이 검정에서 반복 단위는 무엇일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
def cell_level_test(ad, by, group, reference):
    """세포 하나를 관측 하나로 보는 Wilcoxon 검정 (scanpy 기본)"""
    a = ad.copy()
    sc.pp.filter_genes(a, min_cells=10)
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    sc.tl.rank_genes_groups(a, by, groups=[group], reference=reference, method="wilcoxon")
    return sc.get.rank_genes_groups_df(a, group).set_index("names")

mono = kang[kang.obs["cell"] == "CD14+ Monocytes"].copy()
cell_res = cell_level_test(mono, "stim", "stim", "ctrl")
print(f"세포 {mono.n_obs}개, 유전자 {len(cell_res)}개 중 padj < 0.05: {(cell_res.pvals_adj < 0.05).sum()}개")

ISG = ["ISG15", "IFI6", "IFIT1", "MX1"]    # IFN 반응 유전자 = positive control
cell_res.loc[ISG, ["logfoldchanges", "pvals_adj"]]

## 셀 15. Pseudobulk + DESeq2

**프롬프트**: > 기증자-조건별로 count를 합쳐서 DESeq2로 ~donor + stim 비교를 해줘.

**예상**: 샘플은 몇 개가 될까? 설계식은 5주차의 무엇과 같은 자리일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
def pseudobulk(ad, by):
    """by 열의 값마다 세포의 count를 더한다 → 샘플 × 유전자 표"""
    rows = {g: np.asarray(ad[ad.obs[by] == g].X.sum(axis=0)).ravel() for g in sorted(ad.obs[by].unique())}
    df = pd.DataFrame(rows, index=ad.var_names).T.astype(int)
    return df.loc[:, df.sum(axis=0) >= 10]          # 5주차와 같은 거르기

def deseq(counts_df, metadata, design, contrast):
    dds = DeseqDataSet(counts=counts_df, metadata=metadata, design=design, quiet=True)
    dds.deseq2()
    ds = DeseqStats(dds, contrast=contrast, quiet=True)
    ds.summary()
    return ds.results_df

mono.obs["sample"] = mono.obs["ind"] + "_" + mono.obs["stim"]     # 예: 1015_ctrl
counts_df = pseudobulk(mono, "sample")
metadata = pd.DataFrame({"donor": counts_df.index.str.split("_").str[0],
                         "stim": pd.Categorical(counts_df.index.str.split("_").str[1], categories=["ctrl", "stim"])},
                        index=counts_df.index)                      # 기준 수준 ctrl (5주차 5.9절)
print("pseudobulk:", counts_df.shape)

pb_res = deseq(counts_df, metadata, "~donor + stim", ["stim", "stim", "ctrl"])
print(f"padj < 0.05: {(pb_res.padj < 0.05).sum()}개")
pb_res.to_csv(out_dir / "kang_cd14mono_pseudobulk.csv")
pb_res.loc[ISG, ["baseMean", "log2FoldChange", "padj"]]

## 셀 16. 차이가 없어야 하는 비교

**프롬프트**: > 대조군만으로 기증자를 4명씩 나눠 두 방법을 비교해줘.

**예상**: 처리가 없는 비교. 각 방법에서 padj < 0.05인 유전자는 몇 개일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
ctrl = mono[mono.obs["stim"] == "ctrl"].copy()
group_a = sorted(ctrl.obs["ind"].unique())[:4]                    # 기증자 4명 vs 나머지 4명
ctrl.obs["fake"] = np.where(ctrl.obs["ind"].isin(group_a), "A", "B")
print("A:", group_a)

r_cell = cell_level_test(ctrl, "fake", "B", "A")                   # 세포 단위
pb = pseudobulk(ctrl, "ind")                                       # 기증자마다 합 → 샘플 8개
md = pd.DataFrame({"fake": pd.Categorical(np.where(pb.index.isin(group_a), "A", "B"), categories=["A", "B"])},
                  index=pb.index)
r_pb = deseq(pb, md, "~fake", ["fake", "B", "A"])

print(f"세포 단위:  padj < 0.05 {(r_cell.pvals_adj < 0.05).sum()}개 / {len(r_cell)}")
print(f"pseudobulk: padj < 0.05 {(r_pb.padj < 0.05).sum()}개 / {r_pb.padj.notna().sum()}")
kang.write(out_dir / "kang2018_singlets.h5ad")                   # 실습 3.5에서 씀

---
# 다음

- **실습 3.5**: Qoka 대화 입력창에서 프롬프트 레파토리 (강의노트 6.11절). 에이전트가 쓰는 코드는 `analysis/week6/`
- **실습 4**: 커널 재시작 후 전체 실행 (Restart & Run All), 커밋 (강의노트 6.12절)